# XGBoost多因子量化交易策略演示

本笔记本演示如何使用我们开发的XGBoost多因子量化交易策略系统。

## 1. 导入必要的库

In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# 导入自定义模块
from config.settings import Config
from data.data_loader import DataLoader
from data.factor_engine import FactorEngine
from models.feature_engineer import FeatureEngineer
from models.xgboost_model import XGBoostModel
from utils.helpers import load_pickle, load_json

# 设置图表样式
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ 库导入完成")

## 2. 加载已训练的模型

In [ ]:
# 加载模型
model = XGBoostModel()
model.load_model('../data/models/xgboost_model.pkl')

# 加载特征工程器
feature_engineer = FeatureEngineer()
feature_engineer.load_feature_engineer('../data/models/feature_engineer.pkl')

# 加载训练结果
results = load_json('../data/models/training_results.json')

print("✅ 模型加载完成")
print(f"模型参数: {model.params}")

## 3. 分析模型性能

In [ ]:
# 显示测试集性能
print("📊 测试集性能指标:")
for metric, value in results['test_metrics'].items():
    print(f"  {metric}: {value:.4f}")

print("\n📊 交叉验证性能指标:")
cv_metrics = results['cv_result']['avg_metrics']
for metric, value in cv_metrics.items():
    if 'mean_' in metric:
        clean_metric = metric.replace('mean_', '')
        print(f"  {clean_metric}: {value:.4f}")

## 4. 特征重要性分析

In [ ]:
# 获取特征重要性
importance_df = pd.DataFrame(results['feature_importance'])
top_features = importance_df.head(15)

# 绘制特征重要性图
plt.figure(figsize=(12, 8))
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 15 Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("📊 Top 10 重要特征:")
for i, row in top_features.head(10).iterrows():
    print(f"  {i+1}. {row['feature']}: {row['importance']:.2f}")

## 5. 单只股票预测演示

In [ ]:
# 初始化数据加载器和因子引擎
data_loader = DataLoader()
factor_engine = FactorEngine()

# 获取单只股票数据
symbol = '000001.SZ'
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

print(f"📈 获取 {symbol} 数据...")
stock_data = data_loader.get_stock_data(symbol, start_date, end_date)
print(f"数据范围: {stock_data.index[0]} 到 {stock_data.index[-1]}")
print(f"数据条数: {len(stock_data)}")

# 显示价格走势
plt.figure(figsize=(12, 6))
plt.plot(stock_data.index, stock_data['close'], label='收盘价')
plt.title(f'{symbol} 价格走势')
plt.xlabel('日期')
plt.ylabel('价格')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. 因子计算和预测

In [ ]:
# 计算因子
print("🔧 计算技术因子...")
factors = factor_engine.calculate_technical_factors(stock_data)
print(f"计算了 {factors.shape[1]} 个技术因子")

# 预处理因子
factors_processed = factor_engine.preprocess_factors(factors)
factors_processed = factors_processed.dropna()

if len(factors_processed) > 0:
    # 选择数值列
    numeric_columns = factors_processed.select_dtypes(include=[np.number]).columns
    factors_numeric = factors_processed[numeric_columns]
    
    # 应用特征工程（仅缩放，不重新选择特征）
    factors_scaled = feature_engineer.scale_features(factors_numeric, fit=False)
    
    # 选择训练时使用的特征
    selected_features = feature_engineer.selected_features
    available_features = [f for f in selected_features if f in factors_scaled.columns]
    
    if len(available_features) > 10:  # 确保有足够的特征
        factors_final = factors_scaled[available_features]
        
        # 进行预测
        print("🔮 进行预测...")
        predictions = model.predict(factors_final)
        
        # 显示最近的预测结果
        recent_data = factors_final.tail(10)
        recent_predictions = predictions[-10:]
        recent_prices = stock_data['close'].tail(10)
        
        print("\n📊 最近10天的预测结果:")
        for i, (date, pred, price) in enumerate(zip(recent_data.index, recent_predictions, recent_prices)):
            print(f"  {date.strftime('%Y-%m-%d')}: 预测收益率 {pred:.4f}, 当前价格 {price:.2f}")
    else:
        print("⚠️ 可用特征不足，无法进行预测")
else:
    print("⚠️ 因子数据不足，无法进行预测")

## 7. 策略信号生成示例

In [ ]:
# 简单的信号生成逻辑
if 'predictions' in locals() and len(predictions) > 0:
    # 设置信号阈值
    buy_threshold = 0.02   # 预测收益率大于2%时买入
    sell_threshold = -0.02 # 预测收益率小于-2%时卖出
    
    # 生成交易信号
    signals = []
    for pred in predictions:
        if pred > buy_threshold:
            signals.append('买入')
        elif pred < sell_threshold:
            signals.append('卖出')
        else:
            signals.append('持有')
    
    # 统计信号分布
    signal_counts = pd.Series(signals).value_counts()
    print("📊 交易信号分布:")
    for signal, count in signal_counts.items():
        print(f"  {signal}: {count} 次 ({count/len(signals)*100:.1f}%)")
    
    # 显示最近的信号
    print("\n📈 最近10天的交易信号:")
    recent_signals = signals[-10:]
    for i, (date, signal, pred) in enumerate(zip(recent_data.index, recent_signals, recent_predictions)):
        print(f"  {date.strftime('%Y-%m-%d')}: {signal} (预测: {pred:.4f})")
else:
    print("⚠️ 无预测数据，无法生成信号")

## 8. 总结

In [ ]:
print("🎉 XGBoost多因子量化交易策略演示完成!")
print("\n📋 系统特点:")
print("  ✅ 多因子模型: 结合技术指标和基本面指标")
print("  ✅ 机器学习: 使用XGBoost算法进行预测")
print("  ✅ 特征工程: 包含特征选择、交互特征、标准化")
print("  ✅ 风险控制: 时间序列验证避免未来信息泄露")
print("  ✅ 模块化设计: 易于扩展和维护")
print("\n📈 下一步可以:")
print("  1. 开发完整的回测框架")
print("  2. 实现组合优化算法")
print("  3. 添加更多风险控制机制")
print("  4. 连接实盘交易接口")